In [0]:
df = spark.read.table("teams.sensorx.df_sx_800")

#display(df)


In [0]:
## one-hot encoding fyrir PropertiesDeviceID
from pyspark.ml.feature import StringIndexer, OneHotEncoder

df_ohe = spark.read.table("teams.sensorx.df_sx_800")

# Step 1: StringIndexer – maps each unique deviceId string to a numeric index
indexer = StringIndexer(inputCol="properties_deviceId", outputCol="deviceId_index")
df_indexed = indexer.fit(df_ohe).transform(df_ohe)

# Step 2: OneHotEncoder – converts the numeric index to a one-hot sparse vector
encoder = OneHotEncoder(inputCol="deviceId_index", outputCol="deviceId_ohe")
df_encoded = encoder.fit(df_indexed).transform(df_indexed)

do horizon for the failiures - create a new column with failiure horizon based on some horizon H


Altering data for models:

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 0) Config & evaluator
lags_list = [21, 51, 71, 101, 151]  # can adjust
feature_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
]

auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# 1) Base DF (only columns we actually use)
base = (
    df
    .withColumn("timeStamp", F.col("timeStamp").cast("timestamp"))
    .select("timeStamp", "payload_fault_state", *feature_cols)
)

# 2) Global window (single device) — keeps correct chronological order
w = Window.partitionBy("".orderBy(F.col("timeStamp").asc())

# 3) Precompute all lag columns in ONE projection, then persist
maxL = max(lags_list) - 1
lag_exprs = [
    F.lag(feat, L).over(w).alias(f"{feat}{L}")
    for L in range(1, maxL + 1)
    for feat in feature_cols
]

df_all = base.select("timeStamp", "payload_fault_state", *lag_exprs) \
             .persist(StorageLevel.MEMORY_AND_DISK)

_ = df_all.count()  # materialize to cache

# 4) Cutoff using cluster-side percentile on epoch seconds (fast & scalable)
cutoff_epoch = df_all.selectExpr("percentile_approx(CAST(timeStamp AS BIGINT), 0.8)").first()[0]

# 5) Train/evaluate for each lag depth
LR_auc_vector = []

for l in lags_list:
    these_lags = [f"{feat}{L}" for L in range(1, l) for feat in feature_cols]

    # Select only needed columns for this lag depth, and drop initial nulls
    df2_l = (
        df_all
        .select("timeStamp", "payload_fault_state", *these_lags)
        .na.drop(subset=these_lags)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    _ = df2_l.count()  # materialize

    # Assemble once, then split
    assembler = VectorAssembler(inputCols=these_lags, outputCol="features")
    features_df = (
        assembler.transform(df2_l)
        .select("timeStamp", "features", F.col("payload_fault_state").cast("int").alias("label"))
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    _ = features_df.count()

    train_v = features_df.filter(F.col("timeStamp").cast("bigint") <= F.lit(cutoff_epoch)).cache()
    test_v  = features_df.filter(F.col("timeStamp").cast("bigint")  > F.lit(cutoff_epoch)).cache()
    train_v.count(); test_v.count()

    # Logistic Regression — fewer iters + slightly looser tol for speed
    lr = LogisticRegression(
        maxIter=50,              # was 100
        regParam=0.0,
        elasticNetParam=0.0,
        tol=1e-4,                # stop earlier if converged
        standardization=True     # default; keep for stable optimization
    )

    model = lr.fit(train_v)
    preds = model.transform(test_v)
    auc = auc_eval.evaluate(preds)
    LR_auc_vector.append(auc)
    print(f"Lag {l} AUC: {auc}")

    # Clean up per-iteration persists to keep memory healthy
    train_v.unpersist()
    test_v.unpersist()
    features_df.unpersist()
    df2_l.unpersist()

# Cleanup global cache when done
df_all.unpersist()

display(LR_auc_vector)

Logistic Regression with lag:

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# --- 1) Read data (all devices) ---
df_lr = spark.read.table("teams.sensorx.df_sx_800")

# --- 2) One-Hot Encode properties_deviceId ---
indexer = StringIndexer(inputCol="properties_deviceId", outputCol="deviceId_index")
df_indexed = indexer.fit(df_lr).transform(df_lr)

encoder = OneHotEncoder(inputCol="deviceId_index", outputCol="deviceId_ohe")
df_encoded = encoder.fit(df_indexed).transform(df_indexed)

# --- 3) Create lag features (partitioned by device for correctness) ---
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
]
n_lags = 20
w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

for L in range(1, n_lags + 1):
    for col_name in sensor_cols:
        df_encoded = df_encoded.withColumn(f"{col_name}{L}", F.lag(col_name, L).over(w))

lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

# Drop rows with null lags (first n_lags rows per device)
df_clean = df_encoded.na.drop(subset=lag_cols)

# --- 4) Assemble features: sensor cols + OHE vector + lag cols ---
feature_input_cols = sensor_cols + ["deviceId_ohe"] + lag_cols
assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
df_features = assembler.transform(df_clean) \
    .select("timeStamp", "features", F.col("payload_fault_state").cast("int").alias("label"))

# --- 5) Chronological 80/20 train/test split ---
cutoff_epoch = df_features.selectExpr(
    "percentile_approx(CAST(timeStamp AS BIGINT), 0.8)"
).first()[0]

train = df_features.filter(F.col("timeStamp").cast("bigint") <= cutoff_epoch).cache()
test  = df_features.filter(F.col("timeStamp").cast("bigint") >  cutoff_epoch).cache()
print(f"Train: {train.count():,}  |  Test: {test.count():,}")

# --- 6) Train Logistic Regression ---
lr = LogisticRegression(maxIter=100, regParam=0.0, elasticNetParam=0.0)
LRmodel = lr.fit(train)
LRpredictions = LRmodel.transform(test)

# --- 7) Evaluate ---
auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
LRtrainingSummary = LRmodel.summary
LR_auc = auc_eval.evaluate(LRpredictions)
print(f"\nLogistic Regression with OHE + {n_lags} lags")
print(f"AUC: {LR_auc:.4f}")

# Confusion matrix
LRpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

Random forest with lag:

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


# --- 4) RandomForest (Spark ML) ---
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

RFmodel = rf.fit(train)
RFpred = RFmodel.transform(test)

# Extract the summary from the returned RandomForestClassifier instance trained
# in the earlier example
RFtrainingSummary = RFmodel.summary

# Obtain the receiver-operating characteristic as a dataframe and areaUnderROC.
RFtrainingSummary.roc.show()
print("recallByLabel: " + str(RFtrainingSummary.recallByLabel))

print("precisionByLabel: " + str(RFtrainingSummary.precisionByLabel))

RFpred.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

# --- 5) Simple evaluation (F1) ---
eval_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)



Gradient Boosted trees:


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.feature import StringIndexer, VectorIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Index labels, adding metadata to the label column.
labelIndexer = StringIndexer(inputCol="label", outputCol="indexedLabel").fit(train)

# Automatically identify categorical features, and index them.
featureIndexer = VectorIndexer(inputCol="features", outputCol="indexedFeatures", maxCategories=4).fit(train)

# Train a GBT model.
gbt = GBTClassifier(labelCol="indexedLabel", featuresCol="indexedFeatures", maxIter=10)

# Chain indexers and GBT in a Pipeline
pipeline = Pipeline(stages=[labelIndexer, featureIndexer, gbt])

# Train model.  This also runs the indexers.
model = pipeline.fit(train)

# Make predictions.
gbtpredictions = model.transform(test)

# Select example rows to display.
display(gbtpredictions.select("prediction", "indexedLabel", "features").limit(5))

# Select (prediction, true label) and compute test error
evaluator = MulticlassClassificationEvaluator(
    labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(gbtpredictions)
print("Test Error = %g" % (1.0 - accuracy))

gbtModel = model.stages[2]
print(gbtModel)  # summary only

In [0]:
# Logistic Regression:
LRmatrix = (
    LRpredictions.groupBy("label")
        .pivot("prediction")
        .count()
        .orderBy("label")
)
LRmatrix.show()
print("areaunderROC: " + str(LRtrainingSummary.areaUnderROC))

# Random Forest Classifier:
RFmatrix = (
    RFpred.groupBy("label")
        .pivot("prediction")
        .count()
        .orderBy("label")
)
RFmatrix.show()

print("areaunderROC: " + str(RFtrainingSummary.areaUnderROC))

# Gradient Boosted Trees Classifier:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

GBTmatrix = (
    gbtpredictions.groupBy("label")
        .pivot("prediction")
        .count()
        .orderBy("label")
)
GBTmatrix.show()

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",  # or "probability"
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(gbtpredictions)
print("AUC =", auc)

In [0]:


# =====================================================
# 4. Evaluate Random Forest
# =====================================================
print("\nRandom Forest")
RFmatrix = confusion(RFpred)
RFmatrix.show()
RF_auc = auc_eval.evaluate(RFpred)
print("AUC =", RF_auc)

# =====================================================
# 5. Evaluate Gradient Boosted Trees
# =====================================================
print("\nGradient Boosted Trees")
GBTmatrix = confusion(gbtpredictions)
GBTmatrix.show()
GBT_auc = auc_eval.evaluate(gbtpredictions)


print("AUC =", GBT_auc)

In [0]:
# Adding delta t to the table to see how much time has passed between measurements

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read table
df = spark.read.table("teams.sensorx.df_sx_800")

# Ordering by timestamp
order_cols = [F.col("timeStamp").asc()]
if  "serialNumber" and ("serialNumber" in df.columns):
    order_cols.append(F.col("serialNumber").asc())
w = Window.partitionBy("properties_deviceId").orderBy(*order_cols)

# Compute delta column in seconds. If negative then change to NULL
df_with_delta = (
    df
    .withColumn("prev_ts", F.lag("timeStamp").over(w))
    .withColumn(
        "delta_seconds",
        F.when(F.col("prev_ts").isNull(), F.lit(None).cast("double"))
         .otherwise(
             F.when(
                 (F.col("timeStamp").cast("long") - F.col("prev_ts").cast("long")) < 0,
                 None
             ).otherwise(F.col("timeStamp").cast("long") - F.col("prev_ts").cast("long"))
         )
    )
    .drop("prev_ts")
)


df_delta = spark.read.table("teams.sensorx.df_sx_800_with_delta")

display(df_delta.limit(30))

In [0]:
from pyspark.sql import functions as F

table_name = "teams.sensorx.df_sx_800_with_delta"  # or the source table if you haven’t created the new one
device_id  = "3a687fe1-e2bc-4bbf-f64c-08dad2cde7be"                  # e.g., "dev-001234"

df = spark.table(table_name)

display(
    df.where(F.col("properties_deviceId") == F.lit(device_id))
      .orderBy(F.col("timeStamp").asc())
      .limit(200)  # optional
)